In [1]:
# ============================================================
# CEP SEASONAL DISEASE PREDICTION
# FILE 02: DATA CLEANING
# ============================================================

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. Import Libraries
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import os


# ------------------------------------------------------------
# 3. Define Project Paths
# ------------------------------------------------------------

PROJECT_PATH = "/content/drive/MyDrive/CEP Seasonal Disease Prediction"

RAW_PATH = os.path.join(
    PROJECT_PATH,
    "Data",
    "raw"
)

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "Data",
    "processed"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)


# ------------------------------------------------------------
# 4. Load Raw Dataset
# ------------------------------------------------------------

INPUT_FILE = os.path.join(
    RAW_PATH,
    "Final_data.csv"
)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Dataset not found:\n{INPUT_FILE}"
    )

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("CEP SEASONAL DISEASE PREDICTION")
print("02 - DATA CLEANING")
print("=" * 70)

print("\nOriginal Dataset Shape:")
print(df.shape)


# ------------------------------------------------------------
# 5. Make a Copy
# ------------------------------------------------------------

cleaned_df = df.copy()


# ------------------------------------------------------------
# 6. Remove Unnecessary Index Column
# ------------------------------------------------------------

cleaned_df.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
    inplace=True
)


# ------------------------------------------------------------
# 7. Standardize Column Names
# ------------------------------------------------------------

cleaned_df.columns = (
    cleaned_df.columns
    .str.strip()
)


# ------------------------------------------------------------
# 8. Remove Leading / Trailing Spaces
#    From Text Columns
# ------------------------------------------------------------

text_columns = cleaned_df.select_dtypes(
    include=["object"]
).columns

for column in text_columns:

    cleaned_df[column] = (
        cleaned_df[column]
        .astype(str)
        .str.strip()
    )


# ------------------------------------------------------------
# 9. Filter Maharashtra Only
# ------------------------------------------------------------

cleaned_df = cleaned_df[
    cleaned_df["state_ut"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("maharashtra")
].copy()

print("\nAfter Maharashtra Filtering:")
print(cleaned_df.shape)


# ------------------------------------------------------------
# 10. Standardize State Name
# ------------------------------------------------------------

cleaned_df["state_ut"] = "Maharashtra"


# ------------------------------------------------------------
# 11. Standardize District Names
# ------------------------------------------------------------

district_mapping = {

    # Known spelling variations
    "Nasik": "Nashik",
    "Nashik ": "Nashik",

    "Sholapur": "Solapur",
    "Solapur ": "Solapur",

    "Osmanabad (Maharashtra)": "Osmanabad",

}

cleaned_df["district"] = (
    cleaned_df["district"]
    .replace(district_mapping)
)


# ------------------------------------------------------------
# 12. Standardize Disease Names
# ------------------------------------------------------------
# Only obvious duplicate naming variations are combined.
# We do NOT merge diseases that may represent different
# surveillance categories without evidence.

disease_mapping = {

    "Dengue Fever": "Dengue",

    "Dengue Chikungunya":
        "Dengue And Chikungunya",

    "Dengue/Chikungunya":
        "Dengue And Chikungunya",

    "Suspected Dengue And Chikungunya":
        "Suspected Dengue And Chikungunya"

}

cleaned_df["Disease"] = (
    cleaned_df["Disease"]
    .replace(disease_mapping)
)


# ------------------------------------------------------------
# 13. Convert Numerical Columns
# ------------------------------------------------------------

numeric_columns = [
    "year",
    "mon",
    "day",
    "week_of_outbreak",
    "Cases",
    "Deaths",
    "preci",
    "LAI",
    "Temp",
    "Latitude",
    "Longitude"
]

for column in numeric_columns:

    if column in cleaned_df.columns:

        cleaned_df[column] = pd.to_numeric(
            cleaned_df[column],
            errors="coerce"
        )


# ------------------------------------------------------------
# 14. Validate Month Values
# ------------------------------------------------------------

invalid_months = cleaned_df[
    ~cleaned_df["mon"].between(1, 12)
]

print(
    "\nInvalid month records:",
    len(invalid_months)
)


# ------------------------------------------------------------
# 15. Validate Day Values
# ------------------------------------------------------------

invalid_days = cleaned_df[
    ~cleaned_df["day"].between(1, 31)
]

print(
    "Invalid day records:",
    len(invalid_days)
)


# ------------------------------------------------------------
# 16. Remove Invalid Date Records
# ------------------------------------------------------------

cleaned_df["Date"] = pd.to_datetime(
    dict(
        year=cleaned_df["year"],
        month=cleaned_df["mon"],
        day=cleaned_df["day"]
    ),
    errors="coerce"
)

invalid_dates = cleaned_df["Date"].isna().sum()

print(
    "Invalid date records:",
    invalid_dates
)

cleaned_df = cleaned_df[
    cleaned_df["Date"].notna()
].copy()


# ------------------------------------------------------------
# 17. Create Season
# ------------------------------------------------------------

def assign_season(month):

    if month in [3, 4, 5]:
        return "Summer"

    elif month in [6, 7, 8, 9]:
        return "Monsoon"

    elif month in [10, 11]:
        return "Post-Monsoon"

    elif month in [12, 1, 2]:
        return "Winter"

    else:
        return np.nan


cleaned_df["Season"] = (
    cleaned_df["mon"]
    .apply(assign_season)
)


# ------------------------------------------------------------
# 18. Handle Cases
# ------------------------------------------------------------

# Cases are essential for our analysis.
# We do not replace missing cases with zero.

missing_cases = cleaned_df["Cases"].isna().sum()

print(
    "\nMissing Cases:",
    missing_cases
)


# ------------------------------------------------------------
# 19. Handle Negative Cases
# ------------------------------------------------------------

negative_cases = (
    cleaned_df["Cases"] < 0
).sum()

print(
    "Negative Case Values:",
    negative_cases
)

if negative_cases > 0:

    cleaned_df.loc[
        cleaned_df["Cases"] < 0,
        "Cases"
    ] = np.nan


# ------------------------------------------------------------
# 20. Handle Deaths
# ------------------------------------------------------------

# Missing deaths are NOT automatically converted to zero.
# We create an indicator so the information is preserved.

cleaned_df["Deaths_missing"] = (
    cleaned_df["Deaths"].isna().astype(int)
)


# ------------------------------------------------------------
# 21. Check Negative Death Values
# ------------------------------------------------------------

negative_deaths = (
    cleaned_df["Deaths"] < 0
).sum()

print(
    "Negative Death Values:",
    negative_deaths
)

if negative_deaths > 0:

    cleaned_df.loc[
        cleaned_df["Deaths"] < 0,
        "Deaths"
    ] = np.nan


# ------------------------------------------------------------
# 22. Check Climate Variables
# ------------------------------------------------------------

print("\nClimate Variable Summary:")

climate_columns = [
    "preci",
    "LAI",
    "Temp"
]

for column in climate_columns:

    if column in cleaned_df.columns:

        print(
            f"\n{column}:"
        )

        print(
            "Missing:",
            cleaned_df[column].isna().sum()
        )

        print(
            "Minimum:",
            cleaned_df[column].min()
        )

        print(
            "Maximum:",
            cleaned_df[column].max()
        )


# ------------------------------------------------------------
# 23. Remove Exact Duplicate Records
# ------------------------------------------------------------

duplicates_before = (
    cleaned_df.duplicated().sum()
)

print(
    "\nExact duplicate records:",
    duplicates_before
)

cleaned_df = (
    cleaned_df
    .drop_duplicates()
    .reset_index(drop=True)
)

duplicates_after = (
    cleaned_df.duplicated().sum()
)

print(
    "Duplicates after cleaning:",
    duplicates_after
)


# ------------------------------------------------------------
# 24. Sort Dataset
# ------------------------------------------------------------

cleaned_df = cleaned_df.sort_values(
    by=[
        "district",
        "Disease",
        "Date"
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# 25. Reorder Columns
# ------------------------------------------------------------

preferred_columns = [
    "Date",
    "year",
    "mon",
    "day",
    "week_of_outbreak",
    "Season",
    "state_ut",
    "district",
    "Disease",
    "Cases",
    "Deaths",
    "Deaths_missing",
    "Latitude",
    "Longitude",
    "preci",
    "Temp",
    "LAI"
]

existing_columns = [
    column
    for column in preferred_columns
    if column in cleaned_df.columns
]

remaining_columns = [
    column
    for column in cleaned_df.columns
    if column not in existing_columns
]

cleaned_df = cleaned_df[
    existing_columns + remaining_columns
]


# ------------------------------------------------------------
# 26. Missing Value Report
# ------------------------------------------------------------

missing_report = pd.DataFrame({

    "Column":
        cleaned_df.columns,

    "Missing_Values":
        [
            cleaned_df[column].isna().sum()
            for column in cleaned_df.columns
        ],

    "Missing_Percentage":
        [
            cleaned_df[column].isna().mean() * 100
            for column in cleaned_df.columns
        ]

})

missing_report = missing_report.sort_values(
    by="Missing_Values",
    ascending=False
)


# ------------------------------------------------------------
# 27. Save Missing Value Report
# ------------------------------------------------------------

MISSING_REPORT_FILE = os.path.join(
    PROCESSED_PATH,
    "missing_value_report.csv"
)

missing_report.to_csv(
    MISSING_REPORT_FILE,
    index=False
)


# ------------------------------------------------------------
# 28. Save Cleaned Dataset
# ------------------------------------------------------------

OUTPUT_FILE = os.path.join(
    PROCESSED_PATH,
    "Maharashtra_Seasonal_Disease_Cleaned.csv"
)

cleaned_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ------------------------------------------------------------
# 29. Final Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA CLEANING COMPLETED")
print("=" * 70)

print(
    "\nOriginal Dataset Shape:",
    df.shape
)

print(
    "Cleaned Dataset Shape:",
    cleaned_df.shape
)

print(
    "Maharashtra Districts:",
    cleaned_df["district"].nunique()
)

print(
    "Disease Categories:",
    cleaned_df["Disease"].nunique()
)

print(
    "Year Range:",
    cleaned_df["year"].min(),
    "-",
    cleaned_df["year"].max()
)

print(
    "Total Cases:",
    cleaned_df["Cases"].sum()
)

print(
    "Missing Cases:",
    cleaned_df["Cases"].isna().sum()
)

print(
    "Missing Deaths:",
    cleaned_df["Deaths"].isna().sum()
)

print(
    "Missing Temperature:",
    cleaned_df["Temp"].isna().sum()
)

print(
    "Missing Precipitation:",
    cleaned_df["preci"].isna().sum()
)

print(
    "Missing LAI:",
    cleaned_df["LAI"].isna().sum()
)

print("\nCleaned Dataset Saved At:")
print(OUTPUT_FILE)

print("\nMissing Value Report Saved At:")
print(MISSING_REPORT_FILE)

print("\n" + "=" * 70)

Mounted at /content/drive
CEP SEASONAL DISEASE PREDICTION
02 - DATA CLEANING

Original Dataset Shape:
(8985, 15)

After Maharashtra Filtering:
(1195, 14)

Invalid month records: 0
Invalid day records: 0
Invalid date records: 0

Missing Cases: 0
Negative Case Values: 0
Negative Death Values: 0

Climate Variable Summary:

preci:
Missing: 1
Minimum: 7.83e-06
Maximum: 4.814444043

LAI:
Missing: 325
Minimum: 0.0
Maximum: 32.0

Temp:
Missing: 119
Minimum: 277.14
Maximum: 327.728

Exact duplicate records: 12
Duplicates after cleaning: 0

DATA CLEANING COMPLETED

Original Dataset Shape: (8985, 15)
Cleaned Dataset Shape: (1183, 17)
Maharashtra Districts: 35
Disease Categories: 11
Year Range: 2009 - 2022
Total Cases: 53030
Missing Cases: 0
Missing Deaths: 842
Missing Temperature: 117
Missing Precipitation: 1
Missing LAI: 320

Cleaned Dataset Saved At:
/content/drive/MyDrive/CEP Seasonal Disease Prediction/Data/processed/Maharashtra_Seasonal_Disease_Cleaned.csv

Missing Value Report Saved At:
/co